# Data Provenance 2 Workflow Provenance Notebook

_by Max Schröder$^1$, Sascha Genehr$^2$, Tanja Auge$^3$, Susanne Staehlke$^4$, Meike Klettke$^3$ and Frank Krüger$^2$_

$^1$ University Library, University of Rostock, Rostock <br>
$^2$ Data Science and Machine Learning Group, Wismar University of Applied Sciences, Wismar <br>
$^3$ Faculty for Informatics and Data Science, University of Regensburg, Regensburg, <br>
$^4$ Department of Cell Biology, Rostock University Medical Center, Rostock

**Abstract**:
This Jupyter notebook encodes parts of the data analysis that was done for the following article: 

*Staehlke S, Bielfeldt M, Zimmermann J, Gruening M, Barke I, Freitag T, et al.*<br>
Pulsed Electrical Stimulation Affects Osteoblast Adhesion and Calcium Ion<br>
Signaling. Cells. 2022 Aug;11(17):2650. https://doi.org/10.3390/cells11172650.

This notebook demonstrates how Jupyter notebooks can be used to transfer data provenance output into workflow provenance.

### Provenance Graph


In [12]:
import xml.etree.ElementTree as ET
from rdflib import Graph, Namespace, URIRef, BNode, Literal
from rdflib.namespace import RDF, RDFS, XSD

# PROV Namespaces
PROV = Namespace("http://www.w3.org/ns/prov#")
BASE = Namespace("http://protocols.elaine.uni-rostock.de#")
BFO  = Namespace("http://purl.obolibrary.org/obo/bfo.owl")
DWPROV   = Namespace("http://dwprov.elaine.uni-rostock.de#")

In [13]:
def shorten(uri):
    uri = str(uri)
    if "prov#" in uri:
        return "prov:" + uri.split("#")[-1]
    if "rdf-schema#" in uri:
        return "rdfs:" + uri.split("#")[-1]
    if "application#" in uri:
        return ":" + uri.split("#")[-1]
    #if "prov-ext#" in uri:
    #    return uri.split("#")[-1]
    if "rdf-syntax-ns#" in uri:
        return "rdf:" + uri.split("#")[-1]    
    if "elaine.uni-rostock.de#" in uri:
        return "elaine:" + uri.split("#")[-1]  
    return uri

In [14]:
import re

def load_xml(path):
    tree = ET.parse(path)
    return tree.getroot()

def create_graph():
    g = Graph()
    g.bind("xsd", XSD)
    g.bind("prov", PROV)
    g.bind("rdfs", RDFS)
    g.bind("bfo", BFO)
    g.bind("dwprov", DWPROV)
    g.bind("", BASE)
    return g

def add_sources(g, xml_root, only_selected_attrs=False, selected_attrs=None):
    for atom in xml_root.findall(".//originalinstance/atom"):
        source_id = atom.attrib.get("id")

        if not source_id:
            print("⚠️ atom without an ID")
            continue

        source = URIRef(BASE + source_id)

        comment_parts = []
        for const in atom.findall("./constant"):
            attr_name = const.attrib['name']
            attr_value = const.attrib['value']

            if only_selected_attrs and selected_attrs:
                if attr_name not in selected_attrs:
                    continue

            comment_parts.append(f"{attr_name}:{attr_value}")

        
        comment = f" {atom.attrib['name']} (original): " + ", ".join(comment_parts)

        g.add((source, RDF.type, PROV.Entity))
        g.add((source, RDFS.comment, Literal(comment)))

def add_activities(g, xml_root):
    for atom in xml_root.findall(".//provenance/atom"):
        activity_id = atom.attrib['id']
        activity = URIRef(BASE + f"activity_{activity_id}")

        g.add((activity, RDF.type, PROV.Activity))

        # -------------------------
        # WHY (witnesses)
        # -------------------------
        witnesses = []

        for witness in atom.findall("./why/witness_base/witness"):
            #tuple_ids = []
            for tuple_elem in witness.findall("./tuple"):
                tuple_id = tuple_elem.attrib.get("id")
                if tuple_id:
                    witnesses.append(tuple_id)

        if witnesses:
            witness_str = "{" + ",".join(f"{{{tid}}}" for tid in witnesses) + "}"
            why_witness = BNode()
            g.add((why_witness, PROV.value, Literal(witness_str)))
            g.add((activity, DWPROV.hasWhyDataProvenance, why_witness))

        # -------------------------
        # WHERE
        # -------------------------
        for tuple_elem in atom.findall("./where/tuples/tuple"):
            tuple_id = tuple_elem.attrib.get("id")
            if tuple_id:
                where_location = BNode()
                g.add((where_location, PROV.value, Literal(tuple_id)))
                g.add((activity, DWPROV.hasWhereDataProvenance, where_location))
                
                if activity_id.startswith("result_"):
                    used_node = URIRef(BASE + tuple_id)
                    g.add((used_node, RDF.type, PROV.Entity))
                    g.add((activity, PROV.used, used_node))

        # -------------------------
        # HOW
        # -------------------------
        how_exprs = [
            h.attrib['expr']
            for h in atom.findall("./how/polynom")
            if 'expr' in h.attrib
        ]

        if how_exprs:
            how_string = ", ".join(how_exprs)

            how_polynomial = BNode()
            g.add((how_polynomial, PROV.value, Literal(how_string)))
            g.add((activity, DWPROV.hasHowDataProvenance, how_polynomial))

def add_targets(g, xml_root):
    for atom in xml_root.findall(".//instance/atom"):
        target_id = atom.attrib.get('id')

        if not target_id:
            continue

        target = URIRef(BASE + target_id)

        comment_parts = []
        for const in atom.findall("./constant"):
            comment_parts.append(f"{const.attrib['name']}:{const.attrib['value']}")

        comment = f"{atom.attrib['name']}: " + ", ".join(comment_parts)

        g.add((target, RDF.type, PROV.Entity))
        g.add((target, RDFS.comment, Literal(comment)))
        g.add((target, PROV.wasGeneratedBy, URIRef(BASE + f"activity_{target_id}")))

def _extract_region_attrs(query_str: str) -> list[str]:
    region_regex = re.compile(r"SELECT .*? '([^']+)' AS region", re.S | re.I)

    allowed = {"R1_1", "R2_1", "R3_1"}
    regions = set()
    for match in region_regex.finditer(query_str):
        region_literal = match.group(1).strip()          
        if region_literal in allowed:  
            m = re.match(r"R(\d+)_\d+", region_literal)
            if m:
                num = m.group(1)                            
                regions.add(f"r1_{num}")

    return sorted(regions, key=lambda x: int(x.split("_")[1]))

def build_rdf_from_xml(xml_path: str, select: bool, query: str | None = None):
    xml_root = load_xml(xml_path)

    g = create_graph()

    if not select:
        add_sources(g, xml_root, only_selected_attrs=False)
    else:
        if query is None:
            add_sources(g, xml_root, only_selected_attrs=False)
        else:
            region_attrs = _extract_region_attrs(query)

            selected_attrs = ["provid", "time", "type"] + region_attrs

            add_sources(g, xml_root,
                        only_selected_attrs=True,
                        selected_attrs=selected_attrs)
    

    add_activities(g, xml_root)
    add_targets(g, xml_root)

    return g

In [15]:
select = True # print only select attributes

g = build_rdf_from_xml("provenance_output.xml", select, query)

print(f"Number of triples: {len(g)}")

g.serialize("output.ttl", format="ttl")
print("\n✅ RDF saved as output.ttl")

Number of triples: 65

✅ RDF saved as output.ttl


In [24]:
from rdflib import Graph
import subprocess

short = True # shortened namespaces

g = Graph()
g.parse("output.ttl", format="ttl")

dot_file = "graph.dot"
png_file = "graph.png"

with open("graph.dot", "w") as f:
    f.write("digraph G {\n")

    for s, p, o in g:
        s_short = shorten(s) if short else str(s)
        p_short = shorten(p) if short else str(p)
        o_short = shorten(o) if short else str(o)

        f.write(f'    "{s_short}" -> "{o_short}" [label="{p_short}"];\n')

    f.write("}\n")

print("✅ DOT file created")

try:
    subprocess.run(["dot", "-Tpng", dot_file, "-o", png_file], check=True)
    print("✅ PNG file created")
except FileNotFoundError:
    print("❌ Graphviz is not installed (dot is missing)")
except subprocess.CalledProcessError:
    print("❌ Error creating the PNG")

✅ DOT file created


dot: graph is too large for cairo-renderer bitmaps. Scaling by 0.54969 to fit


✅ PNG file created
